# Run static algorithm

This file can be used to run static algorithms as ground truth data to compare DynsbOD data against

In [1]:
from pathlib import Path

# distod_jar = "distod-1.0.6_error.jar"
distod_jar = "distod.jar"
import subprocess
from tqdm import tqdm
def run_distod_on(file: Path):
    # Placeholder for the actual distod function
    result_path = Path('samples') / f'results_{file.name}.txt'
    command = f"""java -Xms2g -Xmx2g -XX:+UseG1GC -Ddistod.input.path="{file.absolute()}" -Ddistod.output.path="{result_path.absolute()}" -Ddistod.input.has-header="yes" -Dfile.encoding=UTF-8 -jar {distod_jar}"""
    try:
        print(command)
        result = subprocess.check_output(command, shell=True, text=True)
        print(result)
        # sort lines and save them to result_path
        output_path = Path("data") / "results.txt"
        result_path.write_text('\n'.join(sorted(output_path.read_text().splitlines())))
    except:
        pass


def run_hyod_on(file: Path, result_path = None):
    if result_path is None:
        result_path = Path('samples') / f'results_hyod_{file.name}.txt'
    # Placeholder for the actual hyod function
    command = f"""java -Xms2g -Xmx16g -jar HyOD_modified.jar '{file.absolute()}' '{result_path.absolute()}'"""
    try:
        print(command)
        process = subprocess.Popen(command, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        output_lines = []
        for line in process.stdout:
            print(line, end='')
            output_lines.append(line)
        process.wait()
        result = ''.join(output_lines)
        lines = result.splitlines()
        memUsed = None
        time = None
        for line in lines:
            if "MemoryCost: " in line:
                memUsed = line.split("MemoryCost: ")[-1]
            if "TotalTime:" in line:
                time = line.split("TotalTime:")[-1]

        return memUsed, time

    except Exception as e:
        print(e)


# HyOD runtime 57s with not full CPU usage (singlecore?) 3min20 on distod
import json
metrics = json.loads(Path('hyod_metrics.json').read_text()) if Path('hyod_metrics.json').exists() else {}

In [4]:
run_hyod_on(Path('../data_before_error_update1379.csv'), Path('../results_before_update.txt'))
run_hyod_on(Path('../data_with_error_update1389.csv'), Path('../results_after_update.txt'))


java -Xms2g -Xmx16g -jar HyOD_modified.jar '/Users/paulsieben/Programming/Masterarbeit/algo/datasets/../data_before_error_update1379.csv' '/Users/paulsieben/Programming/Masterarbeit/algo/datasets/../results_before_update.txt'
HyOD args:
/Users/paulsieben/Programming/Masterarbeit/algo/datasets/../data_before_error_update1379.csv
/Users/paulsieben/Programming/Masterarbeit/algo/datasets/../results_before_update.txt
this file has 93 lines
random sample size: 9
|r'|:9
level 1 start
level 2 start
level 3 start
level 4 start
level 5 start
level 6 start
level 7 start
level 8 start
level 9 start
level 10 start
level 11 start
HyOD Finish
TotalTime:1830ms
sampleTime:6ms, dicoverTime:1705ms, validTime:119ms
MemoryCost: 1017MB
Valid on |r|:8328Valid on |r'|:382650
SampleSize:92
ODs found:8277
java -Xms2g -Xmx16g -jar HyOD_modified.jar '/Users/paulsieben/Programming/Masterarbeit/algo/datasets/../data_with_error_update1389.csv' '/Users/paulsieben/Programming/Masterarbeit/algo/datasets/../results_afte

('946MB', '1828ms')

In [3]:

existing_files = [f.name for f in Path('samples').iterdir() if f.is_file() and f.name.startswith('results_')]
files_to_run_on = [f for f in Path('samples').iterdir() 
                    if f.is_file() 
                        and '_0_' in f.name 
                        and f.suffix == '.csv'
                        and f"results_hyod_{f.name}.txt" not in existing_files
                    ]

files_to_run_on

[PosixPath('samples/flights_20_500k_0_499999.csv'),
 PosixPath('samples/flights_20_500k_0_390000_violating_tuples_first.csv'),
 PosixPath('samples/ncvoter_22_1m_utf8_0_390000_violating_tuples_first.csv')]

In [3]:
# run_distod_on(Path('samples') /"ncvoter_22_1m_utf8_0_390000.csv")
run_distod_on(Path('samples') /"flights_20_500k_0_390000.csv")



java -Xms2g -Xmx2g -XX:+UseG1GC -Ddistod.input.path="/Users/paulsieben/Programming/Masterarbeit/algo/datasets/samples/flights_20_500k_0_390000.csv" -Ddistod.output.path="/Users/paulsieben/Programming/Masterarbeit/algo/datasets/samples/results_flights_20_500k_0_390000.csv.txt" -Ddistod.input.has-header="yes" -Dfile.encoding=UTF-8 -jar distod-assembly-1.0.6-fromgit.jar


SLF4J: A number (1) of logging calls during the initialization phase have been intercepted and are
SLF4J: now being replayed. These are subject to the filtering rules of the underlying logging system.
SLF4J: See also http://www.slf4j.org/codes.html#replay


In [2]:
run_distod_on(Path('crafted/distod_error.csv'))

java -Xms2g -Xmx2g -XX:+UseG1GC -Ddistod.input.path="/Users/paulsieben/Programming/Masterarbeit/algo/datasets/crafted/distod_error.csv" -Ddistod.output.path="/Users/paulsieben/Programming/Masterarbeit/algo/datasets/samples/results_distod_error.csv.txt" -Ddistod.input.has-header="yes" -Dfile.encoding=UTF-8 -jar distod.jar


SLF4J: A number (1) of logging calls during the initialization phase have been intercepted and are
SLF4J: now being replayed. These are subject to the filtering rules of the underlying logging system.
SLF4J: See also http://www.slf4j.org/codes.html#replay


14:15:23.302 [distod-akka.actor.default-dispatcher-3] INFO akka.event.slf4j.Slf4jLogger - Slf4jLogger started
14:15:23.596 [distod-akka.actor.default-dispatcher-3] INFO akka.remote.artery.tcp.ArteryTcpTransport - Remoting started with transport [Artery tcp]; listening on address [akka://distod@127.0.0.1:7878] with UID [-8147515491613666652]
14:15:23.607 [distod-akka.actor.default-dispatcher-3] INFO akka.cluster.Cluster - Cluster Node [akka://distod@127.0.0.1:7878] - Starting up, Akka version [2.6.3] ...
14:15:23.739 [distod-akka.actor.default-dispatcher-3] INFO akka.cluster.Cluster - Cluster Node [akka://distod@127.0.0.1:7878] - Registered cluster JMX MBean [akka:type=Cluster]
14:15:23.739 [distod-akka.actor.default-dispatcher-3] INFO akka.cluster.Cluster - Cluster Node [akka://distod@127.0.0.1:7878] - Started up successfully
14:15:23.774 [distod-akka.actor.default-dispatcher-14] INFO akka.cluster.Cluster - Cluster Node [akka://distod@127.0.0.1:7878] - No downing-provider-class configu

In [4]:



for file in tqdm(files_to_run_on):
    try:
        memUsed, time = run_hyod_on(file)
        metrics[file.name] = {"MemoryCost": memUsed, "TotalTime": time}
    except Exception as e:
        print(e)
        metrics[file.name] = {"MemoryCost": None, "TotalTime": None, "Error": str(e)}

Path("metrics.json").write_text(json.dumps(metrics, indent=4))

NameError: name 'files_to_run_on' is not defined

In [17]:
Path("metrics.json").write_text(json.dumps(metrics, indent=4))


184

In [3]:
# run hyod on wikipedia data

import pandas as pd

for file in tqdm([x for x in Path("wikipedia/dynfd").rglob("**/*.csv") if x.name.startswith('baseline') or x.name.startswith('final_state')]):
    try:
        results_path = file.parent / f'results_hyod_{file.name}.txt'
        # if results_path.exists():
        #     continue
        if not 'disease' in str(file) and not 'cpu' in str(file):
            continue
        pd.read_csv(file).drop(columns=["event_type", "time"], errors="ignore").to_csv(file, index=False)
        memUsed, time = run_hyod_on(file, result_path=results_path)
        # run_distod_on(file)
        metrics[file.name] = {"MemoryCost": memUsed, "TotalTime": time}
    except Exception as e:
        print(e)
        metrics[file.name] = {"MemoryCost": None, "TotalTime": None, "Error": str(e)}


Path("metrics.json").write_text(json.dumps(metrics, indent=4))


  0%|          | 0/8 [00:00<?, ?it/s]

java -Xms2g -Xmx16g -jar HyOD_modified.jar '/Users/paulsieben/Programming/Masterarbeit/algo/datasets/wikipedia/dynfd/disease/final_state.csv' '/Users/paulsieben/Programming/Masterarbeit/algo/datasets/wikipedia/dynfd/disease/results_hyod_final_state.csv.txt'
HyOD args:
/Users/paulsieben/Programming/Masterarbeit/algo/datasets/wikipedia/dynfd/disease/final_state.csv
/Users/paulsieben/Programming/Masterarbeit/algo/datasets/wikipedia/dynfd/disease/results_hyod_final_state.csv.txt
this file has 5759 lines
random sample size: 575
|r'|:293
level 1 start
level 2 start
level 3 start
level 4 start
level 5 start
level 6 start
level 7 start
level 8 start
level 9 start
level 10 start
level 11 start
level 12 start
level 13 start
HyOD Finish
TotalTime:90792ms
sampleTime:42ms, dicoverTime:90301ms, validTime:451ms
MemoryCost: 7330MB
Valid on |r|:366Valid on |r'|:343273
SampleSize:5753
ODs found:364


 12%|█▎        | 1/8 [01:31<10:39, 91.37s/it]

java -Xms2g -Xmx16g -jar HyOD_modified.jar '/Users/paulsieben/Programming/Masterarbeit/algo/datasets/wikipedia/dynfd/disease/baseline.csv' '/Users/paulsieben/Programming/Masterarbeit/algo/datasets/wikipedia/dynfd/disease/results_hyod_baseline.csv.txt'
HyOD args:
/Users/paulsieben/Programming/Masterarbeit/algo/datasets/wikipedia/dynfd/disease/baseline.csv
/Users/paulsieben/Programming/Masterarbeit/algo/datasets/wikipedia/dynfd/disease/results_hyod_baseline.csv.txt
this file has 2881 lines
random sample size: 288
|r'|:247
level 1 start
level 2 start
level 3 start
level 4 start
level 5 start
level 6 start
level 7 start
level 8 start
level 9 start
level 10 start
level 11 start
level 12 start


 25%|██▌       | 2/8 [02:00<05:28, 54.80s/it]

...

 38%|███▊      | 3/8 [02:02<02:33, 30.65s/it]

java -Xms2g -Xmx16g -jar HyOD_modified.jar '/Users/paulsieben/Programming/Masterarbeit/algo/datasets/wikipedia/dynfd/cpu/baseline.csv' '/Users/paulsieben/Programming/Masterarbeit/algo/datasets/wikipedia/dynfd/cpu/results_hyod_baseline.csv.txt'
HyOD args:
/Users/paulsieben/Programming/Masterarbeit/algo/datasets/wikipedia/dynfd/cpu/baseline.csv
/Users/paulsieben/Programming/Masterarbeit/algo/datasets/wikipedia/dynfd/cpu/results_hyod_baseline.csv.txt
this file has 62 lines
random sample size: 6
|r'|:6
level 1 start
level 2 start
level 3 start
level 4 start
level 5 start
level 6 start
level 7 start


100%|██████████| 8/8 [02:03<00:00, 15.39s/it]

level 8 start
level 9 start
level 10 start
HyOD Finish
TotalTime:588ms
sampleTime:6ms, dicoverTime:516ms, validTime:66ms
MemoryCost: 175MB
Valid on |r|:4891Valid on |r'|:145794
SampleSize:61
ODs found:4816


184